# L3-00：环境与工件预检

在真实 PYNQ 板端 Jupyter 中执行 `Restart Kernel and Run All Cells`。

本 Notebook 不加载 overlay，只冻结 Skill/仓库/release 指纹，并证明篡改 manifest 会在下载前被拒绝。
结果写入仓库的 `data/skill/l3_<时间>/00_preflight.json`，同时更新
`data/skill/l3_current_session.json`。

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys

import pynq


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


repo = Path("/home/xilinx/jupyter_notebooks/amd_fpga_2026").resolve()
assert (repo / "board/vivado_zynq7000_xc7z020_pynq/release/pynq.bit").is_file(), (
    "PYNQ release not found under fixed repository path: %s" % repo
)
os.chdir(str(repo))
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "skill/pynq-overlay-workflow/scripts"))

from pynq_skill import artifacts, hwh
from pynq_skill.artifacts import ArtifactError

git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=str(repo), text=True
).strip()
git_status_short = subprocess.check_output(
    ["git", "status", "--short"], cwd=str(repo), text=True
).splitlines()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = repo / "data/skill" / ("l3_" + run_id)
run_dir.mkdir(parents=True, exist_ok=False)
pointer = repo / "data/skill/l3_current_session.json"
write_json(pointer, {"run_id": run_id, "run_dir": str(run_dir)})

skill_file = repo / "skill/pynq-overlay-workflow/SKILL.md"
skill_sha256 = sha256_file(skill_file)
expected_skill_sha256 = os.environ.get(
    "L3_SKILL_SHA256",  # 可选：只有明确要验证另一版 Skill 时才填写 64 位 SHA-256
    "967ccd78278580f96150a8e84177408dd189b265a5e04d1fc31ff45d9ec5f9ad",
).lower()
assert skill_sha256 == expected_skill_sha256, {
    "actual": skill_sha256,
    "expected": expected_skill_sha256,
    "hint": "freeze the intended Skill or set L3_SKILL_SHA256 explicitly",
}

model_path = Path("/proc/device-tree/model")
board_model = (
    model_path.read_bytes().rstrip(b"\x00").decode("utf-8", "replace")
    if model_path.is_file()
    else "not_available"  # 板端缺少 device-tree 时的如实占位；不要手填板型
)
release = repo / "board/vivado_zynq7000_xc7z020_pynq/release"
manifest_path = release / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

art = artifacts.check_artifacts(
    release / "pynq.bit",
    manifest=manifest_path,
    extra=(release / "pynq_svc.bin", release / "pynq_svc.coe", release / "pynq_svc.elf"),
)
info = hwh.parse_hwh(art["hwh"])

bad_manifest = json.loads(json.dumps(manifest))
bad_manifest["artifacts"]["pynq.bit"]["sha256"] = "0" * 64  # 故意篡改，用于证明门禁会失败
try:
    artifacts.check_artifacts(release / "pynq.bit", manifest=bad_manifest)
except ArtifactError as exc:
    negative_hash_gate = {"status": "PASS", "error_type": type(exc).__name__}
else:
    negative_hash_gate = {"status": "FAIL", "reason": "tampered manifest was accepted"}

checks = {
    "skill_hash_frozen": {"status": "PASS", "actual": skill_sha256, "expected": expected_skill_sha256},
    "release_artifacts": {"status": "PASS", "sha256": art["sha256"]},
    "release_manifest_part": {
        "status": "PASS" if manifest.get("part") == "xc7z020clg400-1" else "FAIL",
        "value": manifest.get("part"),
    },
    "release_fclk0_100mhz": {
        "status": "PASS" if info["clocks"].get("fclk0_mhz") == 100.0 else "FAIL",
        "value": info["clocks"].get("fclk0_mhz"),
    },
    "tampered_manifest_rejected": negative_hash_gate,
}
status = "PASS" if all(item["status"] == "PASS" for item in checks.values()) else "FAIL"
record = {
    "case": "preflight",
    "status": status,
    "claim_level": "environment",
    "hardware_touched": False,
    "timestamp_local": datetime.now().isoformat(timespec="seconds"),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "run_id": run_id,
    "run_dir": str(run_dir),
    "board_model": board_model,
    "hostname": platform.node(),
    "python_version": platform.python_version(),
    "pynq_version": pynq.__version__,
    "kernel": platform.platform(),
    "git_commit": git_commit,
    "git_status_short": git_status_short,
    "skill_path": str(skill_file),
    "skill_sha256": skill_sha256,
    "release_manifest": str(manifest_path),
    "release_manifest_sha256": sha256_file(manifest_path),
    "release_artifact_sha256": art["sha256"],
    "release_hwh_ips": {
        name: {
            "phys_addr": item["phys_addr"],
            "addr_range": item["addr_range"],
            "vlnv": item["vlnv"],
        }
        for name, item in info["ips"].items()
    },
    "checks": checks,
    "limitation": "This stage validates environment and fingerprints; it does not load an overlay.",
}
write_json(run_dir / "00_preflight.json", record)
print(json.dumps(record, indent=2, ensure_ascii=False))
assert status == "PASS", checks